# Alternative Encoder Architecture — Contrastive Learning

This notebook swaps in alternative encoder architectures and compares them against the
trained EchoCare-CLIP models from the March 26 experiments.

## Experiments

| # | Image Encoder | Text Encoder | Trainable |
|---|:---:|:---:|:---:|
| 1 | ViT-B/16 (frozen) | CLIP text (frozen) | MLP heads only |
| 2 | ViT-B/16 (fine-tuned) | CLIP text (frozen) | MLP + ViT |
| 3 | EchoCare (frozen) | BERT-base (frozen) | MLP heads only |
| 4 | EchoCare (frozen) | BERT-base (fine-tuned) | MLP + BERT |

## Sections
1. **Configuration** — paths and hyperparameters
2. **Data Loading** — auto-detect CSV columns, verify image paths
3. **Train / Val / Test Split** — stratified by dataset source
4. **Dataset & DataLoaders**
5. **Model Definitions** — ViT image encoder, BERT text encoder, EchoCare, CLIP text, contrastive wrapper
6. **Training Experiments** — four alternative-architecture experiments
7. **Load EchoCare-CLIP Baselines** — load three trained models from Mar 26
8. **Results** — alignment score comparison across all models

---
## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q monai torch torchvision transformers einops matplotlib scikit-learn tqdm pandas Pillow scipy

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPTextModel
from transformers import BertTokenizer, BertModel
from torchvision.models import vit_b_16, ViT_B_16_Weights
from monai.networks.nets.swin_unetr import SwinTransformer

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'CUDA GPU not found — this notebook requires a GPU'
device = torch.device('cuda')
print(f'Using device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Section 1 — Configuration

**Edit all paths and hyperparameters here.** Nothing else in the notebook needs to change.

In [ ]:
config = {
    # ── Paths ──────────────────────────────────────────────────────────────
    'prepared_data_root': '/content/drive/MyDrive/BMI702 Project/ Ultrasound Data/prepared_data',
    'echocare_checkpoint': '/content/drive/MyDrive/BMI702 Project/echocare_encoder.pth',

    # ── Trained EchoCare-CLIP model weights (from Mar 26 experiments) ──────
    'echocare_clip_dir': '/content/drive/MyDrive/BMI702 Project/training_result_epoch20_Mar26',

    # ── Image Encoder (EchoCare SwinTransformer) ───────────────────────────
    'feature_size'   : 128,   # SwinTransformer base channel dim
    'in_channels'    : 3,     # RGB
    'image_size'     : 256,   # H = W

    # ── ViT Image Encoder ──────────────────────────────────────────────────
    'vit_image_size' : 224,   # ViT-B/16 native resolution

    # ── Text Encoder (CLIP) ────────────────────────────────────────────────
    'clip_model_name': 'openai/clip-vit-base-patch32',
    'max_seq_len'    : 77,

    # ── Text Encoder (BERT) ────────────────────────────────────────────────
    'bert_model_name': 'bert-base-uncased',
    'bert_max_seq_len': 128,

    # ── Shared Latent Space ────────────────────────────────────────────────
    'projection_dim' : 256,

    # ── Normalization (computed from train set in Mar 26 notebook) ─────────
    'img_mean': [0.1949, 0.1975, 0.203],
    'img_std' : [0.1951, 0.1974, 0.2025],

    # ── Data splits ────────────────────────────────────────────────────────
    'test_size'  : 0.15,
    'val_size'   : 0.15,
    'random_seed': 42,

    # ── Training ───────────────────────────────────────────────────────────
    'batch_size'      : 32,
    'num_epochs'      : 20,
    'lr'              : 1e-4,
    'weight_decay'    : 1e-4,
    'init_temperature': 0.07,
    'num_workers'     : 2,
}

print('Configuration:')
for k, v in config.items():
    print(f'  {k:30s}: {v}')

---
## Section 2 — Data Loading

Loads the pre-saved train / val / test split DataFrames from the `data_splits/` folder
produced by the March 26 notebook.

In [ ]:
# ── Load pre-saved train / val / test splits from the Mar 26 notebook ──────────
splits_dir = os.path.join(config['prepared_data_root'], 'data_splits')

train_df = pd.read_parquet(os.path.join(splits_dir, 'train_df.parquet'))
val_df   = pd.read_parquet(os.path.join(splits_dir, 'val_df.parquet'))
test_df  = pd.read_parquet(os.path.join(splits_dir, 'test_df.parquet'))

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

total = len(train_df) + len(val_df) + len(test_df)
print(f'Loaded pre-saved splits from: {splits_dir}')
print(f'  Train : {len(train_df):>6,}  ({100*len(train_df)/total:.1f}%)')
print(f'  Val   : {len(val_df):>6,}  ({100*len(val_df)/total:.1f}%)')
print(f'  Test  : {len(test_df):>6,}  ({100*len(test_df)/total:.1f}%)')
print(f'Columns: {list(train_df.columns)}')


---
## Section 4 — Dataset & DataLoaders

In [ ]:
class MultiDatasetUltrasound(Dataset):
    """
    Multi-source ultrasound dataset with free-text captions.

    Returns per sample:
      image   : (3, image_size, image_size) float tensor, normalized
      caption : str  — clinical text description
      dataset : str  — source dataset name

    Transforms:
      train  — Resize(288) → RandomCrop(256) → RandomHorizontalFlip → ToTensor → Normalize
      val/test — Resize(256) → CenterCrop(256) → ToTensor → Normalize
    """

    def __init__(self, dataframe: pd.DataFrame, split: str = 'train',
                 cfg: dict = None, image_size: int = None):
        if cfg is None:
            cfg = config
        self.df    = dataframe.reset_index(drop=True)
        self.split = split

        sz   = image_size if image_size is not None else cfg['image_size']
        mean = cfg['img_mean']
        std  = cfg['img_std']

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize(int(sz * 1.125)),  # 288 for sz=256
                transforms.RandomCrop(sz),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(sz),
                transforms.CenterCrop(sz),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        image = self.transform(image)
        return image, row['caption'], row['dataset']

In [ ]:
# ── DataLoaders at 256×256 (for EchoCare-based experiments) ─────────────────────
train_dataset_256 = MultiDatasetUltrasound(train_df, split='train', image_size=256)
val_dataset_256   = MultiDatasetUltrasound(val_df,   split='val',   image_size=256)
test_dataset_256  = MultiDatasetUltrasound(test_df,  split='test',  image_size=256)

_loader_kw = dict(
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    pin_memory=True,
)

train_loader_256 = DataLoader(train_dataset_256, shuffle=True,  drop_last=True,  **_loader_kw)
val_loader_256   = DataLoader(val_dataset_256,   shuffle=False, drop_last=False, **_loader_kw)
test_loader_256  = DataLoader(test_dataset_256,  shuffle=False, drop_last=False, **_loader_kw)

# ── DataLoaders at 224×224 (for ViT-based experiments) ──────────────────────────
train_dataset_224 = MultiDatasetUltrasound(train_df, split='train', image_size=224)
val_dataset_224   = MultiDatasetUltrasound(val_df,   split='val',   image_size=224)
test_dataset_224  = MultiDatasetUltrasound(test_df,  split='test',  image_size=224)

train_loader_224 = DataLoader(train_dataset_224, shuffle=True,  drop_last=True,  **_loader_kw)
val_loader_224   = DataLoader(val_dataset_224,   shuffle=False, drop_last=False, **_loader_kw)
test_loader_224  = DataLoader(test_dataset_224,  shuffle=False, drop_last=False, **_loader_kw)

print('DataLoaders ready:')
print(f'  256×256  Train: {len(train_loader_256):>4} batches  ({len(train_dataset_256):,} samples)')
print(f'  256×256  Val  : {len(val_loader_256):>4} batches  ({len(val_dataset_256):,} samples)')
print(f'  256×256  Test : {len(test_loader_256):>4} batches  ({len(test_dataset_256):,} samples)')
print(f'  224×224  Train: {len(train_loader_224):>4} batches  ({len(train_dataset_224):,} samples)')
print(f'  224×224  Val  : {len(val_loader_224):>4} batches  ({len(val_dataset_224):,} samples)')
print(f'  224×224  Test : {len(test_loader_224):>4} batches  ({len(test_dataset_224):,} samples)')

---
## Section 5 — Model Definitions

Four encoder classes + a generic contrastive wrapper:

- **EchoCareImageEncoder** — SwinTransformer backbone (from Mar 26)
- **ViTImageEncoder** — torchvision ViT-B/16 backbone
- **CLIPTextEncoder** — CLIP text tower
- **BERTTextEncoder** — BERT-base text tower
- **ContrastiveCLIP** — generic wrapper (symmetric InfoNCE + learnable temperature)

In [ ]:
class EchoCareImageEncoder(nn.Module):
    """
    EchoCare SwinTransformer backbone + MLP projection head.

    Input  : (B, 3, 256, 256)
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        feature_size   : int  = 128,
        in_channels    : int  = 3,
        projection_dim : int  = 256,
        use_checkpoint : bool = True,
        freeze_encoder : bool = False,
    ):
        super().__init__()

        self.encoder = SwinTransformer(
            in_chans       = in_channels,
            embed_dim      = feature_size,
            window_size    = [8, 8],
            patch_size     = [2, 2],
            depths         = [2, 2, 18, 2],
            num_heads      = [4, 8, 16, 32],
            mlp_ratio      = 4.0,
            qkv_bias       = True,
            use_checkpoint = use_checkpoint,
            spatial_dims   = 2,
            use_v2         = True,
        )

        encoder_out_dim = feature_size * (2 ** 4)  # 128 * 16 = 2048

        self.projection = nn.Sequential(
            nn.Linear(encoder_out_dim, encoder_out_dim),
            nn.ReLU(),
            nn.Linear(encoder_out_dim, projection_dim),
        )

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def load_pretrained(self, checkpoint_path: str, device: str = 'cpu'):
        """Load EchoCare MAE pretrained weights (removes mask_token artifact)."""
        state_dict = torch.load(checkpoint_path, map_location=device)
        state_dict.pop('mask_token', None)
        self.encoder.load_state_dict(state_dict, strict=True)
        print(f"Loaded EchoCare pretrained weights from '{checkpoint_path}'")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.encoder(x)           # list of feature maps
        deep  = feats[-1]                 # (B, 2048, 8, 8)
        emb   = deep.mean(dim=(2, 3))     # GAP → (B, 2048)
        emb   = self.projection(emb)      # (B, projection_dim)
        return F.normalize(emb, dim=-1)

In [ ]:
class ViTImageEncoder(nn.Module):
    """
    ViT-B/16 backbone (ImageNet-pretrained) + MLP projection head.

    Input  : (B, 3, 224, 224)
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        projection_dim : int  = 256,
        freeze_encoder : bool = False,
    ):
        super().__init__()

        self.encoder = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        encoder_out_dim = self.encoder.heads.head.in_features  # 768
        self.encoder.heads = nn.Identity()  # remove classification head

        self.projection = nn.Sequential(
            nn.Linear(encoder_out_dim, encoder_out_dim),
            nn.ReLU(),
            nn.Linear(encoder_out_dim, projection_dim),
        )

        if freeze_encoder:
            for name, param in self.encoder.named_parameters():
                param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.encoder(x)             # (B, 768)
        emb = self.projection(emb)        # (B, projection_dim)
        return F.normalize(emb, dim=-1)

In [ ]:
class CLIPTextEncoder(nn.Module):
    """
    CLIP text tower + MLP projection head.

    Input  : (B, 77) token ids
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        clip_model_name     : str  = 'openai/clip-vit-base-patch32',
        projection_dim      : int  = 256,
        freeze_text_encoder : bool = True,
    ):
        super().__init__()

        self.text_encoder = CLIPTextModel.from_pretrained(clip_model_name)
        text_out_dim      = self.text_encoder.config.hidden_size  # 512

        if freeze_text_encoder:
            for param in self.text_encoder.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(text_out_dim, text_out_dim),
            nn.ReLU(),
            nn.Linear(text_out_dim, projection_dim),
        )

    def forward(
        self,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        outputs  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = outputs.pooler_output          # (B, 512) — [EOS] token embedding
        text_emb = self.projection(text_emb)      # (B, projection_dim)
        return F.normalize(text_emb, dim=-1)

In [ ]:
class BERTTextEncoder(nn.Module):
    """
    BERT-base text tower + MLP projection head.

    Input  : (B, max_seq_len) token ids + attention_mask
    Output : (B, projection_dim)  L2-normalized
    """

    def __init__(
        self,
        bert_model_name     : str  = 'bert-base-uncased',
        projection_dim      : int  = 256,
        freeze_text_encoder : bool = True,
    ):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained(bert_model_name)
        text_out_dim      = self.text_encoder.config.hidden_size  # 768

        if freeze_text_encoder:
            for param in self.text_encoder.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(text_out_dim, text_out_dim),
            nn.ReLU(),
            nn.Linear(text_out_dim, projection_dim),
        )

    def forward(
        self,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        outputs  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = outputs.pooler_output          # (B, 768) — [CLS] token embedding
        text_emb = self.projection(text_emb)      # (B, projection_dim)
        return F.normalize(text_emb, dim=-1)

In [ ]:
class ContrastiveCLIP(nn.Module):
    """
    Generic CLIP-style contrastive model.
    Wraps any image_encoder and text_encoder that both produce (B, D) L2-normalized embeddings.

    Training objective: symmetric InfoNCE loss over matched (image, text) pairs.
    Temperature: learnable log_temperature, initialized at log(init_temperature).
    """

    def __init__(
        self,
        image_encoder,
        text_encoder,
        init_temperature: float = 0.07,
    ):
        super().__init__()
        self.image_encoder   = image_encoder
        self.text_encoder    = text_encoder
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temperature)))

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        return self.image_encoder(images)

    def encode_text(self, input_ids, attention_mask=None) -> torch.Tensor:
        return self.text_encoder(input_ids, attention_mask)

    def forward(self, images, input_ids, attention_mask=None):
        image_emb = self.encode_image(images)                    # (B, D)
        text_emb  = self.encode_text(input_ids, attention_mask)  # (B, D)

        temperature      = self.log_temperature.exp()
        logits_per_image = (image_emb @ text_emb.T) / temperature  # (B, B)
        logits_per_text  = logits_per_image.T

        labels   = torch.arange(image_emb.size(0), device=image_emb.device)
        loss_i2t = F.cross_entropy(logits_per_image, labels)
        loss_t2i = F.cross_entropy(logits_per_text,  labels)
        loss     = (loss_i2t + loss_t2i) / 2

        return loss, image_emb, text_emb

In [ ]:
# ── Also define EchoCare_CLIP (same as Mar 26) for loading saved baselines ──────
class EchoCare_CLIP(nn.Module):
    """
    CLIP-style contrastive model combining EchoCare image encoder and CLIP text encoder.
    Needed to load the Mar 26 trained weights.
    """

    def __init__(
        self,
        image_encoder   : EchoCareImageEncoder,
        text_encoder    : CLIPTextEncoder,
        init_temperature: float = 0.07,
    ):
        super().__init__()
        self.image_encoder   = image_encoder
        self.text_encoder    = text_encoder
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temperature)))

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        return self.image_encoder(images)

    def encode_text(self, input_ids, attention_mask=None) -> torch.Tensor:
        return self.text_encoder(input_ids, attention_mask)

    def forward(self, images, input_ids, attention_mask=None):
        image_emb = self.encode_image(images)
        text_emb  = self.encode_text(input_ids, attention_mask)

        temperature      = self.log_temperature.exp()
        logits_per_image = (image_emb @ text_emb.T) / temperature
        logits_per_text  = logits_per_image.T

        labels   = torch.arange(image_emb.size(0), device=image_emb.device)
        loss_i2t = F.cross_entropy(logits_per_image, labels)
        loss_t2i = F.cross_entropy(logits_per_text,  labels)
        loss     = (loss_i2t + loss_t2i) / 2

        return loss, image_emb, text_emb

In [ ]:
def tokenize_clip(texts, tokenizer, max_length=77, device='cpu'):
    """
    Tokenize a list of strings with the CLIP tokenizer.
    Returns (input_ids, attention_mask) both shape (B, max_length).
    """
    encoded = tokenizer(
        texts,
        padding        = 'max_length',
        max_length     = max_length,
        truncation     = True,
        return_tensors = 'pt',
    )
    return (
        encoded['input_ids'].to(device),
        encoded['attention_mask'].to(device),
    )


def tokenize_bert(texts, tokenizer, max_length=128, device='cpu'):
    """
    Tokenize a list of strings with the BERT tokenizer.
    Returns (input_ids, attention_mask) both shape (B, max_length).
    """
    encoded = tokenizer(
        texts,
        padding        = 'max_length',
        max_length     = max_length,
        truncation     = True,
        return_tensors = 'pt',
    )
    return (
        encoded['input_ids'].to(device),
        encoded['attention_mask'].to(device),
    )


clip_tokenizer = CLIPTokenizer.from_pretrained(config['clip_model_name'])
bert_tokenizer = BertTokenizer.from_pretrained(config['bert_model_name'])
print(f'CLIP tokenizer loaded  |  vocab_size = {clip_tokenizer.vocab_size:,}')
print(f'BERT tokenizer loaded  |  vocab_size = {bert_tokenizer.vocab_size:,}')

---
## Section 6 — Training Experiments

Four alternative-architecture experiments:

| # | Image Encoder | Text Encoder | Trainable |
|---|:---:|:---:|:---:|
| 1 | ViT-B/16 **frozen** | CLIP text **frozen** | MLP heads only |
| 2 | ViT-B/16 **fine-tuned** | CLIP text **frozen** | MLP + ViT |
| 3 | EchoCare **frozen** | BERT-base **frozen** | MLP heads only |
| 4 | EchoCare **frozen** | BERT-base **fine-tuned** | MLP + BERT |

All experiments use: AdamW optimizer · CosineAnnealingLR · best-checkpoint selection.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment infrastructure
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, dataloader, optimizer, tokenize_fn, tokenizer, device, epoch):
    model.train()
    total_loss = 0.0
    for batch_idx, (images, texts, _) in enumerate(dataloader):
        images = images.to(device)
        ids, mask = tokenize_fn(texts, tokenizer, device=device)
        optimizer.zero_grad()
        loss, _, _ = model(images, ids, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 10 == 0 or batch_idx == len(dataloader) - 1:
            print(f'  [E{epoch+1}] Batch {batch_idx+1}/{len(dataloader)}'
                  f' | loss={loss.item():.4f}'
                  f' | temp={model.log_temperature.exp().item():.4f}')
    return total_loss / len(dataloader)


@torch.no_grad()
def validate_one_epoch(model, dataloader, tokenize_fn, tokenizer, device):
    model.eval()
    total_loss = 0.0
    for images, texts, _ in dataloader:
        images = images.to(device)
        ids, mask = tokenize_fn(texts, tokenizer, device=device)
        loss, _, _ = model(images, ids, mask)
        total_loss += loss.item()
    return total_loss / len(dataloader)


@torch.no_grad()
def compute_alignment_score(model, dataloader, tokenize_fn, tokenizer, device):
    """
    Alignment score = mean cosine similarity between matched (image, text) pairs.
    Both embeddings are L2-normalized, so dot product == cosine similarity.
    Range: [-1, 1].  Higher is better.
    """
    model.eval()
    all_img, all_txt = [], []
    for images, texts, _ in dataloader:
        images = images.to(device)
        ids, mask = tokenize_fn(texts, tokenizer, device=device)
        all_img.append(model.encode_image(images).cpu())
        all_txt.append(model.encode_text(ids, mask).cpu())
    img_embs = torch.cat(all_img)  # (N, D)
    txt_embs = torch.cat(all_txt)  # (N, D)

    paired_sim = (img_embs * txt_embs).sum(dim=1).mean().item()  # diagonal

    all_sim   = img_embs @ txt_embs.T
    off_mask  = ~torch.eye(img_embs.size(0), dtype=torch.bool)
    cross_sim = all_sim[off_mask].mean().item()

    print(f'  Alignment score (paired):   {paired_sim:.4f}')
    print(f'  Cross-pair mean similarity: {cross_sim:.4f}')
    return paired_sim


def train_experiment(model, train_loader, val_loader, tokenize_fn, tokenizer,
                     checkpoint_path, label=''):
    """Full training loop. Returns (train_losses, val_losses)."""
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['lr'], weight_decay=config['weight_decay'],
    )
    scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['num_epochs']
    )
    train_losses, val_losses = [], []
    best_val      = float('inf')

    print(f'\n{"="*62}')
    print(f'  Training: {label}')
    print(f'{"="*62}')

    for epoch in range(config['num_epochs']):
        t_loss = train_one_epoch(model, train_loader, optimizer,
                                 tokenize_fn, tokenizer, device, epoch)
        v_loss = validate_one_epoch(model, val_loader,
                                    tokenize_fn, tokenizer, device)
        scheduler.step()
        train_losses.append(t_loss)
        val_losses.append(v_loss)
        lr_now = scheduler.get_last_lr()[0]
        print(f'>>> Epoch {epoch+1:2d}/{config["num_epochs"]}'
              f'  Train={t_loss:.4f}  Val={v_loss:.4f}  LR={lr_now:.2e}')
        if v_loss < best_val:
            best_val = v_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f'  [Checkpoint] best val={best_val:.4f} \u2192 {checkpoint_path}')

    # Load best checkpoint before returning
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f'\nTraining complete.  Best val loss: {best_val:.4f}')
    return train_losses, val_losses


print('Experiment infrastructure defined.')

In [ ]:
# ── Helper: build ViT + CLIP-text model ────────────────────────────────────────
def build_vit_clip(freeze_vit=True, freeze_text=True):
    """ViT-B/16 image encoder + CLIP text encoder."""
    img_enc = ViTImageEncoder(
        projection_dim = config['projection_dim'],
        freeze_encoder = freeze_vit,
    )
    txt_enc = CLIPTextEncoder(
        clip_model_name     = config['clip_model_name'],
        projection_dim      = config['projection_dim'],
        freeze_text_encoder = freeze_text,
    )
    model = ContrastiveCLIP(
        image_encoder    = img_enc,
        text_encoder     = txt_enc,
        init_temperature = config['init_temperature'],
    ).to(device)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)')
    return model


# ── Helper: build EchoCare + BERT model ────────────────────────────────────────
def build_echocare_bert(freeze_image=True, freeze_text=True):
    """EchoCare image encoder (pretrained, frozen) + BERT text encoder."""
    img_enc = EchoCareImageEncoder(
        feature_size   = config['feature_size'],
        in_channels    = config['in_channels'],
        projection_dim = config['projection_dim'],
        freeze_encoder = freeze_image,
    )
    if os.path.exists(config['echocare_checkpoint']):
        img_enc.load_pretrained(config['echocare_checkpoint'], device=str(device))
    else:
        print('[WARNING] EchoCare checkpoint not found — random image encoder weights')

    txt_enc = BERTTextEncoder(
        bert_model_name     = config['bert_model_name'],
        projection_dim      = config['projection_dim'],
        freeze_text_encoder = freeze_text,
    )
    model = ContrastiveCLIP(
        image_encoder    = img_enc,
        text_encoder     = txt_enc,
        init_temperature = config['init_temperature'],
    ).to(device)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)')
    return model


# ── Helper: build EchoCare_CLIP (for loading Mar 26 baselines) ──────────────────
def build_echocare_clip(freeze_image=True, freeze_text=True):
    """EchoCare image encoder + CLIP text encoder (same architecture as Mar 26)."""
    img_enc = EchoCareImageEncoder(
        feature_size   = config['feature_size'],
        in_channels    = config['in_channels'],
        projection_dim = config['projection_dim'],
        freeze_encoder = freeze_image,
    )
    if os.path.exists(config['echocare_checkpoint']):
        img_enc.load_pretrained(config['echocare_checkpoint'], device=str(device))
    else:
        print('[WARNING] EchoCare checkpoint not found — random image encoder weights')

    txt_enc = CLIPTextEncoder(
        clip_model_name     = config['clip_model_name'],
        projection_dim      = config['projection_dim'],
        freeze_text_encoder = freeze_text,
    )
    model = EchoCare_CLIP(
        image_encoder    = img_enc,
        text_encoder     = txt_enc,
        init_temperature = config['init_temperature'],
    ).to(device)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)')
    return model


print('Model builders defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 1 — ViT-B/16 frozen + CLIP text frozen → MLP heads only
# ─────────────────────────────────────────────────────────────────────────────
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
os.makedirs(save_dir, exist_ok=True)
_ckpt1 = os.path.join(save_dir, 'alt1_vit_frozen_clip_frozen_best.pth')
_res1  = os.path.join(save_dir, 'alt1_results.npz')

print('Building Exp 1 model (ViT frozen + CLIP text frozen → MLP only)...')
model_alt1 = build_vit_clip(freeze_vit=True, freeze_text=True)

if os.path.exists(_ckpt1) and os.path.exists(_res1):
    print(f'  [SKIP] Checkpoint found, loading saved results from {_res1}')
    model_alt1.load_state_dict(torch.load(_ckpt1, map_location=device))
    _d = np.load(_res1)
    losses_alt1_train = list(_d['train_losses'])
    losses_alt1_val   = list(_d['val_losses'])
    score_alt1        = float(_d['score'])
    print(f'  Loaded: score_alt1={score_alt1:.4f}')
else:
    # ── Device check ──────────────────────────────────────────────────────
    _model_device = next(model_alt1.parameters()).device
    _sample_imgs, _, _ = next(iter(train_loader_224))
    _sample_imgs = _sample_imgs.to(device)
    assert _model_device.type == 'cuda', f'Model is on {_model_device}, expected cuda'
    assert _sample_imgs.device.type == 'cuda', f'Data is on {_sample_imgs.device}, expected cuda'
    print(f'  [OK] Model on: {_model_device}')
    print(f'  [OK] Data  on: {_sample_imgs.device}  shape={tuple(_sample_imgs.shape)}')
    del _sample_imgs
    # ──────────────────────────────────────────────────────────────────────

    losses_alt1_train, losses_alt1_val = train_experiment(
        model_alt1, train_loader_224, val_loader_224,
        tokenize_fn     = tokenize_clip,
        tokenizer       = clip_tokenizer,
        checkpoint_path = _ckpt1,
        label           = 'Alt 1: ViT frozen + CLIP text frozen (MLP only)',
    )

    print('\nAlt 1 alignment score on test set:')
    score_alt1 = compute_alignment_score(
        model_alt1, test_loader_224, tokenize_clip, clip_tokenizer, device)

    np.savez(_res1, train_losses=losses_alt1_train, val_losses=losses_alt1_val, score=score_alt1)
    print(f'  Results saved to {_res1}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 2 — ViT-B/16 fine-tuned + CLIP text frozen → MLP + ViT
# ─────────────────────────────────────────────────────────────────────────────
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
os.makedirs(save_dir, exist_ok=True)
_ckpt2 = os.path.join(save_dir, 'alt2_vit_unfrozen_clip_frozen_best.pth')
_res2  = os.path.join(save_dir, 'alt2_results.npz')

print('Building Exp 2 model (ViT unfrozen + CLIP text frozen → MLP + ViT)...')
model_alt2 = build_vit_clip(freeze_vit=False, freeze_text=True)

if os.path.exists(_ckpt2) and os.path.exists(_res2):
    print(f'  [SKIP] Checkpoint found, loading saved results from {_res2}')
    model_alt2.load_state_dict(torch.load(_ckpt2, map_location=device))
    _d = np.load(_res2)
    losses_alt2_train = list(_d['train_losses'])
    losses_alt2_val   = list(_d['val_losses'])
    score_alt2        = float(_d['score'])
    print(f'  Loaded: score_alt2={score_alt2:.4f}')
else:
    # ── Device check ──────────────────────────────────────────────────────
    _model_device = next(model_alt2.parameters()).device
    _sample_imgs, _, _ = next(iter(train_loader_224))
    _sample_imgs = _sample_imgs.to(device)
    assert _model_device.type == 'cuda', f'Model is on {_model_device}, expected cuda'
    assert _sample_imgs.device.type == 'cuda', f'Data is on {_sample_imgs.device}, expected cuda'
    print(f'  [OK] Model on: {_model_device}')
    print(f'  [OK] Data  on: {_sample_imgs.device}  shape={tuple(_sample_imgs.shape)}')
    del _sample_imgs
    # ──────────────────────────────────────────────────────────────────────

    losses_alt2_train, losses_alt2_val = train_experiment(
        model_alt2, train_loader_224, val_loader_224,
        tokenize_fn     = tokenize_clip,
        tokenizer       = clip_tokenizer,
        checkpoint_path = _ckpt2,
        label           = 'Alt 2: ViT fine-tuned + CLIP text frozen (MLP + ViT)',
    )

    print('\nAlt 2 alignment score on test set:')
    score_alt2 = compute_alignment_score(
        model_alt2, test_loader_224, tokenize_clip, clip_tokenizer, device)

    np.savez(_res2, train_losses=losses_alt2_train, val_losses=losses_alt2_val, score=score_alt2)
    print(f'  Results saved to {_res2}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 3 — EchoCare frozen + BERT frozen → MLP heads only
# ─────────────────────────────────────────────────────────────────────────────
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
os.makedirs(save_dir, exist_ok=True)
_ckpt3 = os.path.join(save_dir, 'alt3_echocare_frozen_bert_frozen_best.pth')
_res3  = os.path.join(save_dir, 'alt3_results.npz')

print('Building Exp 3 model (EchoCare frozen + BERT frozen → MLP only)...')
model_alt3 = build_echocare_bert(freeze_image=True, freeze_text=True)

if os.path.exists(_ckpt3) and os.path.exists(_res3):
    print(f'  [SKIP] Checkpoint found, loading saved results from {_res3}')
    model_alt3.load_state_dict(torch.load(_ckpt3, map_location=device))
    _d = np.load(_res3)
    losses_alt3_train = list(_d['train_losses'])
    losses_alt3_val   = list(_d['val_losses'])
    score_alt3        = float(_d['score'])
    print(f'  Loaded: score_alt3={score_alt3:.4f}')
else:
    # ── Device check ──────────────────────────────────────────────────────
    _model_device = next(model_alt3.parameters()).device
    _sample_imgs, _, _ = next(iter(train_loader_256))
    _sample_imgs = _sample_imgs.to(device)
    assert _model_device.type == 'cuda', f'Model is on {_model_device}, expected cuda'
    assert _sample_imgs.device.type == 'cuda', f'Data is on {_sample_imgs.device}, expected cuda'
    print(f'  [OK] Model on: {_model_device}')
    print(f'  [OK] Data  on: {_sample_imgs.device}  shape={tuple(_sample_imgs.shape)}')
    del _sample_imgs
    # ──────────────────────────────────────────────────────────────────────

    losses_alt3_train, losses_alt3_val = train_experiment(
        model_alt3, train_loader_256, val_loader_256,
        tokenize_fn     = tokenize_bert,
        tokenizer       = bert_tokenizer,
        checkpoint_path = _ckpt3,
        label           = 'Alt 3: EchoCare frozen + BERT frozen (MLP only)',
    )

    print('\nAlt 3 alignment score on test set:')
    score_alt3 = compute_alignment_score(
        model_alt3, test_loader_256, tokenize_bert, bert_tokenizer, device)

    np.savez(_res3, train_losses=losses_alt3_train, val_losses=losses_alt3_val, score=score_alt3)
    print(f'  Results saved to {_res3}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 4 — EchoCare frozen + BERT fine-tuned → MLP + BERT
# ─────────────────────────────────────────────────────────────────────────────
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
os.makedirs(save_dir, exist_ok=True)
_ckpt4 = os.path.join(save_dir, 'alt4_echocare_frozen_bert_unfrozen_best.pth')
_res4  = os.path.join(save_dir, 'alt4_results.npz')

print('Building Exp 4 model (EchoCare frozen + BERT unfrozen → MLP + BERT)...')
model_alt4 = build_echocare_bert(freeze_image=True, freeze_text=False)

if os.path.exists(_ckpt4) and os.path.exists(_res4):
    print(f'  [SKIP] Checkpoint found, loading saved results from {_res4}')
    model_alt4.load_state_dict(torch.load(_ckpt4, map_location=device))
    _d = np.load(_res4)
    losses_alt4_train = list(_d['train_losses'])
    losses_alt4_val   = list(_d['val_losses'])
    score_alt4        = float(_d['score'])
    print(f'  Loaded: score_alt4={score_alt4:.4f}')
else:
    # ── Device check ──────────────────────────────────────────────────────
    _model_device = next(model_alt4.parameters()).device
    _sample_imgs, _, _ = next(iter(train_loader_256))
    _sample_imgs = _sample_imgs.to(device)
    assert _model_device.type == 'cuda', f'Model is on {_model_device}, expected cuda'
    assert _sample_imgs.device.type == 'cuda', f'Data is on {_sample_imgs.device}, expected cuda'
    print(f'  [OK] Model on: {_model_device}')
    print(f'  [OK] Data  on: {_sample_imgs.device}  shape={tuple(_sample_imgs.shape)}')
    del _sample_imgs
    # ──────────────────────────────────────────────────────────────────────

    losses_alt4_train, losses_alt4_val = train_experiment(
        model_alt4, train_loader_256, val_loader_256,
        tokenize_fn     = tokenize_bert,
        tokenizer       = bert_tokenizer,
        checkpoint_path = _ckpt4,
        label           = 'Alt 4: EchoCare frozen + BERT fine-tuned (MLP + BERT)',
    )

    print('\nAlt 4 alignment score on test set:')
    score_alt4 = compute_alignment_score(
        model_alt4, test_loader_256, tokenize_bert, bert_tokenizer, device)

    np.savez(_res4, train_losses=losses_alt4_train, val_losses=losses_alt4_val, score=score_alt4)
    print(f'  Results saved to {_res4}')


In [ ]:
# ── Best checkpoints are already saved to save_dir during training above. ──────
# This cell is kept as a confirmation step.
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
print('All model checkpoints and results already saved to:', save_dir)
import os
for f in sorted(os.listdir(save_dir)):
    print(' ', f)


---
## Section 7 — Load EchoCare-CLIP Baselines (Mar 26)

Load the three trained EchoCare-CLIP models and compute their alignment scores on the
same test set for a fair comparison.

In [ ]:
echocare_clip_weights = {
    'EC-CLIP Exp1\nMLP only': 'echocare_clip_mlp_heads_only.pt',
    'EC-CLIP Exp2\nMLP+ImgEnc': 'echocare_clip_mlp_and_image_encoder.pt',
    'EC-CLIP Exp3\nMLP+TxtEnc': 'echocare_clip_mlp_and_text_encoder.pt',
}

echocare_clip_scores = {}

for label, weight_file in echocare_clip_weights.items():
    weight_path = os.path.join(config['echocare_clip_dir'], weight_file)
    print(f'\nLoading {label.replace(chr(10), " ")} from {weight_path}')

    # Build with all encoders frozen (we only need inference)
    model = build_echocare_clip(freeze_image=True, freeze_text=True)
    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()

    score = compute_alignment_score(
        model, test_loader_256, tokenize_clip, clip_tokenizer, device)
    echocare_clip_scores[label] = score

print('\nAll EchoCare-CLIP baseline scores loaded.')

---
## Section 8 — Results

**Alignment score** = mean cosine similarity between paired (image, text) test embeddings.  
Higher → image and text embeddings are closer in the shared latent space.

In [ ]:
save_dir = '/content/drive/MyDrive/BMI702 Project/training_result_alt_arch_Apr7/'
os.makedirs(save_dir, exist_ok=True)

# ── Collect all results ────────────────────────────────────────────────────────
results = {}

# Alternative architecture experiments (this notebook)
results['Alt 1\nViT+CLIP\n(MLP only)']       = score_alt1
results['Alt 2\nViT+CLIP\n(MLP+ViT)']        = score_alt2
results['Alt 3\nEchoCare+BERT\n(MLP only)']   = score_alt3
results['Alt 4\nEchoCare+BERT\n(MLP+BERT)']   = score_alt4

# EchoCare-CLIP baselines (loaded from Mar 26)
results.update(echocare_clip_scores)

labels = list(results.keys())
scores = list(results.values())

# Colors: alt experiments in warm tones, EchoCare-CLIP baselines in cool tones
bar_colors = [
    '#E07B39',  # Alt 1
    '#C44E52',  # Alt 2
    '#8172B2',  # Alt 3
    '#937860',  # Alt 4
    '#4C72B0',  # EC-CLIP Exp1
    '#DD8452',  # EC-CLIP Exp2
    '#55A868',  # EC-CLIP Exp3
]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(labels, scores, color=bar_colors, edgecolor='white', linewidth=1.2, zorder=3)

# Annotate bar heights
for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{score:.3f}',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
    )

ax.set_ylabel('Mean Paired Cosine Similarity (Alignment Score)', fontsize=11)
ax.set_title('Alternative Architectures vs EchoCare-CLIP Baselines', fontsize=13)
ax.set_ylim(min(0, min(scores) - 0.05), max(scores) * 1.15)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
plt.xticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'alt_vs_echocare_comparison.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print('\nSummary:')
for label, score in results.items():
    lbl = label.replace('\n', ' ')
    print(f'  {lbl:40s}: {score:.4f}')

In [ ]:
# ── Training loss curves for all four alternative experiments ───────────────────
exp_curves = [
    ('Alt 1: ViT+CLIP (MLP only)',     losses_alt1_train, losses_alt1_val, '#E07B39'),
    ('Alt 2: ViT+CLIP (MLP+ViT)',      losses_alt2_train, losses_alt2_val, '#C44E52'),
    ('Alt 3: EchoCare+BERT (MLP only)', losses_alt3_train, losses_alt3_val, '#8172B2'),
    ('Alt 4: EchoCare+BERT (MLP+BERT)', losses_alt4_train, losses_alt4_val, '#937860'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (title, t_losses, v_losses, color) in zip(axes, exp_curves):
    epochs = range(1, len(t_losses) + 1)
    ax.plot(epochs, t_losses, color=color, linewidth=2,               label='Train')
    ax.plot(epochs, v_losses, color=color, linewidth=2, linestyle='--',
            alpha=0.8, label='Val')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('InfoNCE Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Curves — Alternative Architectures', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'alt_training_curves.png'),
            dpi=300, bbox_inches='tight')
plt.show()